<a href="https://colab.research.google.com/github/immaximo/Hybrid-ML-LLM-Tutoring-Model-for-Cloud-Certification-Preparation-/blob/main/DKT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
!pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch_geometric

In [ ]:
import pandas as pd
import numpy as np
import torch

from torch_geometric.data import Data

# =========================
# STEP 1: LOAD DATA
# =========================
interactions = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/synchronized_interaction_logs.csv")          # user_id, question_id, timestamp, correct
questions = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/CDL_150_Questions_With_ID.xlsx")      # question_id, skill_id
googlequestions = pd.read_json("/content/drive/MyDrive/Colab Notebooks/google_cloud_quiz_eval.json")      # question_id, skill_id
studentquiz = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/student_quiz_dataset.csv")      # user_id, skill_id, pre_score, post_score

# Rename 'Question_ID' to 'question_id' in the questions DataFrame to match interactions
questions = questions.rename(columns={'Question_ID': 'question_id'})

# Assign 'Module' as 'skill_id' in the questions DataFrame, assuming Module represents the skill.
# If 'skill_id' is located in a different column, please adjust this line accordingly.
questions['skill_id'] = questions['Module']

# Rename 'student_id' to 'user_id' in interactions DataFrame
interactions = interactions.rename(columns={'student_id': 'user_id'})

# Rename 'student_id' to 'user_id' in studentquiz for merging
studentquiz = studentquiz.rename(columns={'student_id': 'user_id'})

# Merge interactions with studentquiz to get the 'status' (correctness)
# Use a left merge to keep all interactions and add status if available
interactions = interactions.merge(studentquiz[['user_id', 'question_id', 'status']],
                                  on=['user_id', 'question_id'],
                                  how='left')

# Convert 'status' to numerical 'correct' column (1 for 'correct', 0 for 'incorrect')
# Fill NaN values (for interactions without a matching status) with 0, marking them as incorrect by default
interactions['correct'] = interactions['status'].map({'correct': 1, 'incorrect': 0}).fillna(0).astype(int)

# Drop the original 'status' column as it's no longer needed
interactions = interactions.drop(columns=['status'], errors='ignore')

# =========================
# STEP 2: MERGE INTERACTIONS WITH SKILLS
# =========================
data = interactions.merge(questions, on="question_id", how="inner")
data = data.sort_values(by=["user_id", "timestamp"])

# Convert timestamp to datetime objects for calculating elapsed time
data["timestamp"] = pd.to_datetime(data["timestamp"])

# Encode IDs
user2idx = {u: i for i, u in enumerate(data["user_id"].unique())}
q2idx = {q: i for i, q in enumerate(data["question_id"].unique())}
s2idx = {s: i for i, s in enumerate(data["skill_id"].unique())}

data["user_id"] = data["user_id"].map(user2idx)
data["question_id"] = data["question_id"].map(q2idx)
data["skill_id"] = data["skill_id"].map(s2idx)

num_users = len(user2idx)
num_skills = len(s2idx)

# =========================
# STEP 3: BUILD STUDENT SEQUENCES
# =========================
student_sequences = []
grouped = data.groupby("user_id")

for user, group in grouped:
    timestamps = group["timestamp"].values
    # Calculate time difference in seconds between consecutive interactions for each student
    # The first element will be 0 as there's no preceding interaction.
    time_diffs_seconds = np.insert(np.diff(timestamps).astype('timedelta64[s]').astype(float), 0, 0.0)

    student_sequences.append({
        "student_id": user,
        "q_seq": torch.tensor(group["question_id"].values, dtype=torch.long),
        "s_seq": torch.tensor(group["skill_id"].values, dtype=torch.long),
        "r_seq": torch.tensor(group["correct"].values, dtype=torch.float),
        "dt_seq": torch.tensor(time_diffs_seconds, dtype=torch.float) # New: elapsed time in seconds
    })

# =========================
# STEP 4: BUILD SKILL GRAPH
# =========================
adj_matrix = np.zeros((num_skills, num_skills))
for seq in student_sequences:
    skills = seq["s_seq"].numpy()
    for i in range(len(skills)-1):
        s1, s2 = skills[i], skills[i+1]
        adj_matrix[s1, s2] += 1
        adj_matrix[s2, s1] += 1

# Convert to PyG edge_index
edges = np.array(np.nonzero(adj_matrix))
edge_index = torch.tensor(edges, dtype=torch.long)
edge_weight = torch.tensor(adj_matrix[edges[0], edges[1]], dtype=torch.float)

# Node features: one-hot for skills
x = torch.eye(num_skills, dtype=torch.float)

# =========================
# STEP 5: CREATE PyG DATA OBJECT
# =========================
data_gnn = Data(x=x, edge_index=edge_index, edge_attr=edge_weight)
data_gnn.student_sequences = student_sequences

# =========================
# STEP 6: SAVE DATA
# =========================
torch.save(data_gnn, "kt_all_models_prepost.pt")
print("Preprocessing complete. Dataset ready for all models (AKT, DKT, BKT, GRKT/GNN).")

In [ ]:
import torch.nn as nn
!pip install torch_geometric # Keep for now, as it was in original cell. Might not be strictly needed for DKT but not harmful.
# from torch_geometric.nn import GCNConv # Removed as GCNConv is specific to GRKT
import requests

# Fetch DKT model from GitHub
dkt_url = "https://raw.githubusercontent.com/pykt-team/pykt-toolkit/main/pykt/models/dkt.py"
response = requests.get(dkt_url)
response.raise_for_status() # Raise an exception for bad status codes

with open("dkt.py", "w") as f:
    f.write(response.text)

from dkt import DKT

# The GRKT model class definition has been removed as per user request.
# Now, we will instantiate the DKT model in a separate cell.

In [ ]:
num_questions = len(q2idx) # This represents the maximum index for question IDs
num_concepts = len(s2idx)   # This represents the number of unique skills/concepts

emb_size = 128     # Embedding size
hidden_size = 128  # Hidden layer size (not used by DKT from pykt, but good to keep if for other models)
dropout_rate = 0.5 # Dropout rate

# When emb_type is "qid", the DKT model uses the first argument (num_c) to represent the number of questions.
# The internal self.num_c will be used for both interaction embedding size and the output linear layer size.
# Since we use `num_questions` as a padding value in the DataLoader, the model's output layer
# needs to have `num_questions + 1` dimensions to safely accommodate this index.
dkt_model = DKT(num_questions + 1, emb_size, dropout_rate, 'qid')

print("DKT model instantiated successfully!")

In [ ]:
from torch.utils.data import Dataset, DataLoader

class DKTDataset(Dataset):
    def __init__(self, student_sequences):
        self.student_sequences = student_sequences

    def __len__(self):
        return len(self.student_sequences)

    def __getitem__(self, idx):
        seq = self.student_sequences[idx]
        q_seq = seq['q_seq']
        s_seq = seq['s_seq'] # Although DKT often doesn't use skill_seq directly, we'll keep it for consistency or potential future use
        r_seq = seq['r_seq']

        # DKT typically predicts the next response based on past interactions.
        # So we create shifted sequences for input and target.
        # Input: q_seq[:-1], r_seq[:-1]
        # Target: q_seq[1:], r_seq[1:] (for loss calculation against predictions)

        # If a sequence has only one interaction, it cannot be used for sequence prediction
        if len(q_seq) < 2:
            return None # This will be filtered out by collate_fn

        input_q = q_seq[:-1]
        input_r = r_seq[:-1]
        target_q = q_seq[1:] # Question ID for the next step
        target_r = r_seq[1:] # Correctness for the next step

        # The DKT model expects q, r, qshft, rshft
        # q and r represent the history up to time t
        # qshft and rshft represent the elements at time t+1 (the ones to predict)
        return input_q, input_r, target_q, target_r

def collate_fn(batch):
    batch = [item for item in batch if item is not None] # Filter out sequences shorter than 2
    if not batch:
        return None, None, None, None

    input_q, input_r, target_q, target_r = zip(*batch)

    # Use num_questions as padding value for question IDs to avoid masking valid question ID 0.
    # For responses (0 or 1), 0 can be used as padding, assuming masked during loss calculation.
    global num_questions

    input_q = torch.nn.utils.rnn.pad_sequence(input_q, batch_first=True, padding_value=num_questions)
    input_r = torch.nn.utils.rnn.pad_sequence(input_r, batch_first=True, padding_value=0)
    target_q = torch.nn.utils.rnn.pad_sequence(target_q, batch_first=True, padding_value=num_questions)
    target_r = torch.nn.utils.rnn.pad_sequence(target_r, batch_first=True, padding_value=0)

    return input_q, input_r, target_q, target_r

# Create Dataset and DataLoader
dataset = DKTDataset(data_gnn.student_sequences) # Assuming data_gnn.student_sequences is accessible
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

print(f"DKTDataset created with {len(dataset)} sequences.")
print(f"DataLoader created with batch size {batch_size}.")

In [ ]:
from sklearn.metrics import roc_auc_score, mean_squared_error
import numpy as np

# =========================
# STEP 8: TRAINING THE DKT MODEL
# =========================

optimizer = optim.Adam(dkt_model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss() # For binary classification (correct/incorrect)

num_epochs = 20 # Increased number of epochs to 20

dkt_model.train()
print("Starting DKT model training...")

for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
        if input_q is None: # Skip empty batches from collate_fn
            continue

        optimizer.zero_grad()

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)
        dkt_model.to(device)

        # DKT model takes q and r as inputs. Ensure they are long tensors for the embedding layer.
        y_pred_all = dkt_model(input_q.long(), input_r.long())

        # Gather the specific predictions for the target questions
        # y_pred_all usually has shape (batch_size, seq_len, num_c)
        y_pred = y_pred_all.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)

        # Mask out padded items using the new padding_value (num_questions)
        mask = (target_q != num_questions)

        # Calculate loss only on valid (unpadded) steps
        loss = criterion(y_pred[mask], target_r[mask].float())

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print("DKT model training complete.")

In [ ]:
# =========================
# STEP 9: EVALUATE THE DKT MODEL
# =========================

dkt_model.eval()
all_preds = []
all_targets = []

print("Starting DKT model evaluation...")

with torch.no_grad():
    for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
        if input_q is None: # Skip empty batches
            continue

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

        # Get predictions (ensure long tensors)
        y_pred_all = dkt_model(input_q.long(), input_r.long())

        # Gather predictions for the specific target questions
        y_pred = y_pred_all.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)

        # Apply sigmoid to get probabilities
        y_pred_prob = torch.sigmoid(y_pred)

        # Mask out padding values using the new padding_value (num_questions)
        mask = (target_q != num_questions)

        all_preds.extend(y_pred_prob[mask].cpu().numpy())
        all_targets.extend(target_r[mask].cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Calculate AUC
auc = roc_auc_score(all_targets, all_preds)
print(f"DKT Model AUC: {auc:.4f}")

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
print(f"DKT Model RMSE: {rmse:.4f}")

print("DKT model evaluation complete.")

In [ ]:
!pip install optuna

In [ ]:
import optuna
from sklearn.metrics import roc_auc_score
import torch.nn as nn
import torch.optim as optim

def objective(trial):
    # Hyperparameters to tune
    emb_size = trial.suggest_categorical('emb_size', [64, 128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    # Instantiate DKT model with trial hyperparameters
    global num_questions
    model = DKT(num_questions + 1, emb_size, dropout_rate, 'qid')
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # Training loop for a reduced number of epochs for faster tuning
    num_tuning_epochs = 5 # Use fewer epochs for tuning to speed up the process
    model.train()
    for epoch in range(num_tuning_epochs):
        total_loss = 0
        for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
            if input_q is None:
                continue

            optimizer.zero_grad()
            input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

            y_pred_all = model(input_q.long(), input_r.long())
            y_pred = y_pred_all.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)

            mask = (target_q != num_questions)
            loss = criterion(y_pred[mask], target_r[mask].float())

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # Evaluation loop
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
            if input_q is None:
                continue
            input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

            y_pred_all = model(input_q.long(), input_r.long())
            y_pred = y_pred_all.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
            y_pred_prob = torch.sigmoid(y_pred)

            mask = (target_q != num_questions)

            all_preds.extend(y_pred_prob[mask].cpu().numpy())
            all_targets.extend(target_r[mask].cpu().numpy())

    auc = roc_auc_score(all_targets, all_preds)
    return auc

print("Objective function defined for Optuna tuning.")

In [ ]:
# Diagnostic: Check if dkt.py exists in /content
!ls /content/dkt.py

In [ ]:
# Ensures sam-pytorch is installed. The import will happen in cell bf1fe500.
!pip install sam-pytorch

In [ ]:
import sys
sys.path.insert(0, '/content') # Ensure /content is in sys.path to find dkt.py

import optuna
from sklearn.metrics import roc_auc_score
import torch.nn as nn
import torch.optim as optim
from dkt import DKT # Import DKT here to ensure it's in scope for objective
from sam_pytorch.sam import SAM # Import SAM here to ensure it's in scope for objective

def objective(trial):
    # Hyperparameters to tune
    emb_size = trial.suggest_categorical('emb_size', [64, 128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    rho = trial.suggest_float('rho', 0.05, 0.2) # SAM parameter

    # Instantiate DKT model with trial hyperparameters
    global num_questions
    model = DKT(num_questions + 1, emb_size, dropout_rate, 'qid')

    # Use SAM optimizer
    base_optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    optimizer = SAM(base_optimizer, model, rho=rho, adaptive=True)

    criterion = nn.BCEWithLogitsLoss()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # Training loop for a reduced number of epochs for faster tuning
    num_tuning_epochs = 5 # Use fewer epochs for tuning to speed up the process
    model.train()
    for epoch in range(num_tuning_epochs):
        total_loss = 0
        for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
            if input_q is None:
                continue

            input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

            # First forward-backward pass
            predictions = model(input_q.long(), input_r.long())
            y_pred = predictions.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
            mask = (target_q != num_questions)
            loss = criterion(y_pred[mask], target_r[mask].float())
            loss.backward()
            optimizer.first_step(zero_grad=True)

            # Second forward-backward pass
            predictions = model(input_q.long(), input_r.long())
            y_pred = predictions.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
            loss = criterion(y_pred[mask], target_r[mask].float())
            loss.backward()
            optimizer.second_step(zero_grad=True)

            total_loss += loss.item()

    # Evaluation loop
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
            if input_q is None:
                continue
            input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

            y_pred_all = model(input_q.long(), input_r.long())
            y_pred = y_pred_all.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
            y_pred_prob = torch.sigmoid(y_pred)

            mask = (target_q != num_questions)

            all_preds.extend(y_pred_prob[mask].cpu().numpy())
            all_targets.extend(target_r[mask].cpu().numpy())

    auc = roc_auc_score(all_targets, all_preds)
    return auc

print("Objective function defined for Optuna tuning with SAM.")

In [ ]:
# Reinstall sam-pytorch to ensure it's properly installed
!pip install --force-reinstall sam-pytorch

import sys
sys.path.insert(0, '/content') # Ensure /content is in sys.path to find dkt.py

import optuna
from sklearn.metrics import roc_auc_score
import torch.nn as nn
import torch.optim as optim
from dkt import DKT # Import DKT here to ensure it's in scope for objective
from sam_pytorch.sam import SAM # Import SAM here to ensure it's in scope for objective

def objective(trial):
    # Hyperparameters to tune
    emb_size = trial.suggest_categorical('emb_size', [64, 128, 256])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    rho = trial.suggest_float('rho', 0.05, 0.2) # SAM parameter

    # Instantiate DKT model with trial hyperparameters
    global num_questions
    model = DKT(num_questions + 1, emb_size, dropout_rate, 'qid')

    # Use SAM optimizer
    base_optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    optimizer = SAM(base_optimizer, model, rho=rho, adaptive=True)

    criterion = nn.BCEWithLogitsLoss()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # Training loop for a reduced number of epochs for faster tuning
    num_tuning_epochs = 5 # Use fewer epochs for tuning to speed up the process
    model.train()
    for epoch in range(num_tuning_epochs):
        total_loss = 0
        for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
            if input_q is None:
                continue

            input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

            # First forward-backward pass
            predictions = model(input_q.long(), input_r.long())
            y_pred = predictions.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
            mask = (target_q != num_questions)
            loss = criterion(y_pred[mask], target_r[mask].float())
            loss.backward()
            optimizer.first_step(zero_grad=True)

            # Second forward-backward pass
            predictions = model(input_q.long(), input_r.long())
            y_pred = predictions.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
            loss = criterion(y_pred[mask], target_r[mask].float())
            loss.backward()
            optimizer.second_step(zero_grad=True)

            total_loss += loss.item()

    # Evaluation loop
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
            if input_q is None:
                continue
            input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

            y_pred_all = model(input_q.long(), input_r.long())
            y_pred = y_pred_all.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
            y_pred_prob = torch.sigmoid(y_pred)

            mask = (target_q != num_questions)

            all_preds.extend(y_pred_prob[mask].cpu().numpy())
            all_targets.extend(target_r[mask].cpu().numpy())

    auc = roc_auc_score(all_targets, all_preds)
    return auc

print("Objective function defined for Optuna tuning with SAM.")

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # Run 20 trials for hyperparameter optimization

print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

best_params = trial.params

In [ ]:
# Retrain the DKT model with the best hyperparameters found by Optuna

emb_size = best_params['emb_size']
dropout_rate = best_params['dropout_rate']
learning_rate = best_params['learning_rate']
rho = best_params['rho'] # SAM parameter

# Instantiate DKT model with best hyperparameters
dkt_model = DKT(num_questions + 1, emb_size, dropout_rate, 'qid')

# Use SAM optimizer with the base Adam optimizer and weight_decay
base_optimizer = optim.Adam(dkt_model.parameters(), lr=learning_rate, weight_decay=1e-5)
optimizer = SAM(base_optimizer, dkt_model, rho=rho, adaptive=True)

# Added learning rate scheduler to lower learning rate when loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer.base_optimizer, mode='min', factor=0.5, patience=3) # Scheduler should wrap the base optimizer

criterion = nn.BCEWithLogitsLoss()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dkt_model.to(device)

# Train for more epochs to allow further convergence
num_epochs = 40
dkt_model.train()
print(f"Starting DKT model training with best hyperparameters (SAM enabled): emb_size={emb_size}, dropout_rate={dropout_rate}, learning_rate={learning_rate}, rho={rho}...")

for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
        if input_q is None:
            continue

        input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

        # First forward-backward pass (compute gradients for perturbed weights)
        predictions = dkt_model(input_q.long(), input_r.long())
        y_pred = predictions.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
        mask = (target_q != num_questions)
        loss = criterion(y_pred[mask], target_r[mask].float())
        loss.backward()
        optimizer.first_step(zero_grad=True)

        # Second forward-backward pass (update actual weights)
        predictions = dkt_model(input_q.long(), input_r.long())
        y_pred = predictions.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
        loss = criterion(y_pred[mask], target_r[mask].float())
        loss.backward()
        optimizer.second_step(zero_grad=True)

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    scheduler.step(avg_loss) # Update scheduler based on the loss

    # Print every epoch to see the progress
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print("DKT model training with best hyperparameters (SAM enabled) complete.")

In [ ]:
# Retrain the DKT model with the best hyperparameters found by Optuna

emb_size = best_params['emb_size']
dropout_rate = best_params['dropout_rate']
learning_rate = best_params['learning_rate']

# Instantiate DKT model with best hyperparameters
dkt_model = DKT(num_questions + 1, emb_size, dropout_rate, 'qid')

# Added weight_decay for L2 regularization
optimizer = optim.Adam(dkt_model.parameters(), lr=learning_rate, weight_decay=1e-5)

# Added learning rate scheduler to lower learning rate when loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

criterion = nn.BCEWithLogitsLoss()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dkt_model.to(device)

# Train for more epochs to allow further convergence
num_epochs = 40
dkt_model.train()
print(f"Starting DKT model training with best hyperparameters: emb_size={emb_size}, dropout_rate={dropout_rate}, learning_rate={learning_rate}...")

for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
        if input_q is None:
            continue

        optimizer.zero_grad()
        input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

        y_pred_all = dkt_model(input_q.long(), input_r.long())
        y_pred = y_pred_all.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)

        mask = (target_q != num_questions)
        loss = criterion(y_pred[mask], target_r[mask].float())

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    scheduler.step(avg_loss) # Update scheduler based on the loss

    # Print every epoch to see the progress
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print("DKT model training with best hyperparameters complete.")

In [ ]:
# Evaluate the retrained DKT model

dkt_model.eval()
all_preds = []
all_targets = []

print("Starting DKT model evaluation with best hyperparameters...")

with torch.no_grad():
    for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
        if input_q is None:
            continue

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

        y_pred_all = dkt_model(input_q.long(), input_r.long())
        y_pred = y_pred_all.gather(-1, target_q.unsqueeze(-1).long()).squeeze(-1)
        y_pred_prob = torch.sigmoid(y_pred)

        mask = (target_q != num_questions)

        all_preds.extend(y_pred_prob[mask].cpu().numpy())
        all_targets.extend(target_r[mask].cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

auc = roc_auc_score(all_targets, all_preds)
rmse = np.sqrt(mean_squared_error(all_targets, all_preds))

print(f"DKT Model (tuned) AUC: {auc:.4f}")
print(f"DKT Model (tuned) RMSE: {rmse:.4f}")

print("DKT model evaluation with best hyperparameters complete.")

In [ ]:
import time

# Define the target original question IDs requested by the user
target_original_q_ids = [5, 10, 11] # Using available question IDs

print("Calculating detailed latency and predictions per question...")

dkt_model.eval() # Ensure model is in evaluation mode

results = []

with torch.no_grad():
    for original_q_id in target_original_q_ids:
        # 1. Get encoded question ID
        if original_q_id not in q2idx:
            print(f"Warning: Original Question ID {original_q_id} not found in q2idx. Skipping.")
            continue
        encoded_q_id = q2idx[original_q_id]

        # 2. Get original skill ID from the 'questions' DataFrame
        # Assuming 'question_id' in 'questions' df refers to the original_q_id
        original_skill_id = questions[questions['question_id'] == original_q_id]['skill_id'].iloc[0]

        # 3. Get encoded skill ID
        encoded_skill_id = s2idx[original_skill_id]

        # Create dummy input for a single prediction
        # We'll simulate a previous padded interaction and then predict for the current question
        # DKT predicts the next item given the current sequence. So, input_q/r = previous context
        # target_q = the question we want to predict for in the *next* step.

        # A minimal sequence of length 1 for input, and target for that input
        # Using num_questions (padding value) as a dummy previous interaction for consistent input shape
        input_q_tensor = torch.tensor([[num_questions]], dtype=torch.long)
        input_r_tensor = torch.tensor([[0]], dtype=torch.long) # Dummy response (e.g., incorrect)
        target_q_tensor = torch.tensor([[encoded_q_id]], dtype=torch.long)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        input_q_tensor, input_r_tensor, target_q_tensor = (
            input_q_tensor.to(device),
            input_r_tensor.to(device),
            target_q_tensor.to(device)
        )
        dkt_model.to(device)

        start_time = time.time() # Start timing for this specific prediction
        y_pred_all = dkt_model(input_q_tensor, input_r_tensor)
        end_time = time.time() # End timing

        latency_ms = (end_time - start_time) * 1000 # Convert to milliseconds

        # Extract the prediction for the specific question
        # y_pred_all will be (batch_size, sequence_length, num_questions)
        # For our dummy input, it's (1, 1, num_questions)
        y_pred = y_pred_all.gather(-1, target_q_tensor.unsqueeze(-1)).squeeze(-1) # [1, 1]
        predicted_probability = torch.sigmoid(y_pred).item()

        results.append({
            "Question ID": original_q_id,
            "Encoded Skill ID": encoded_skill_id,
            "Predicted Probability (Correct)": predicted_probability,
            "Latency": latency_ms
        })

print("Target Questions and their Encoded Skill IDs:")
for original_q_id in target_original_q_ids:
    if original_q_id in q2idx:
        # Get original skill ID and then encoded skill ID
        original_skill_id = questions[questions['question_id'] == original_q_id]['skill_id'].iloc[0]
        encoded_skill_id = s2idx[original_skill_id]
        print(f"  {{'{original_q_id}': {encoded_skill_id}}}")

print("\nLatency Test Results:")
for res in results:
    print(f"Question ID: {res['Question ID']}")
    print(f"  Encoded Skill ID: {res['Encoded Skill ID']}")
    print(f"  Predicted Probability (Correct): {res['Predicted Probability (Correct)']:.4f}")
    print(f"  Latency: {res['Latency']:.4f} ms")

print("Detailed latency calculation complete.")

In [ ]:
import time
import numpy as np

print("Calculating DKT model prediction latency...")

dkt_model.eval() # Ensure model is in evaluation mode
all_latencies = [] # To store latency measurements

with torch.no_grad():
    for batch_idx, (input_q, input_r, target_q, target_r) in enumerate(dataloader):
        if input_q is None: # Skip empty batches
            continue

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        input_q, input_r, target_q, target_r = input_q.to(device), input_r.to(device), target_q.to(device), target_r.to(device)

        start_time = time.time() # Start timing
        # Only measure the time for the forward pass, not the post-processing
        _ = dkt_model(input_q.long(), input_r.long())
        end_time = time.time() # End timing

        latency_ms = (end_time - start_time) * 1000 # Convert to milliseconds
        all_latencies.append(latency_ms)

mean_latency = np.mean(all_latencies) # Calculate average latency

print(f"Average DKT Model Prediction Latency: {mean_latency:.4f} ms")
print("Latency calculation complete.")